# 🎵 IsaiCraft — Remote GPU Processing Engine (Tier 3)

**An Intelligent Audio Processing Ecosystem for AI-Powered Music Production**

This notebook serves as the remote, cloud-accelerated **Tier 3 GPU Processing Node** for IsaiCraft. It receives vocal recordings and music metadata from the frontend via an authenticated HTTP tunnel, runs generative instrumental synthesis, executes DSP noise reduction and PyWorld soft-pitch correction, performs studio mastering, and returns Base64-encoded audio stems to the client.

---
### 🏗️ Architecture Pipeline
1. **Client Audio Capture**: React frontend records raw vocal audio via Web Audio API.
2. **Neural Music Generation**: Meta MusicGen (`facebook/musicgen-small`) generates conditioned instrumental backing tracks.
3. **Acoustic Cleansing**: `noisereduce` strips hardware static and room noise.
4. **Humanized Pitch Correction**: `pyworld` (DIO + StoneMask + CheapTrick + D4C) applies fractional retuning (0.55 retune strength) to preserve natural human vibrato.
5. **Vocal & Mastering Chain**: `pedalboard` parametric EQ, compression, and reverb.
6. **Dynamic Mixbus**: Soft-clipping (`np.tanh`) and peak balancing (0.35 beat, 0.90 vocal) exported as Base64 JSON.

In [ ]:
# @title 1. Verify GPU Availability
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    !nvidia-smi
else:
    print("WARNING: GPU is not active. Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.")

In [ ]:
# @title 2. Install Core Dependencies
!pip install -q transformers accelerate scipy librosa pyworld noisereduce pedalboard pyngrok fastapi uvicorn python-multipart nest-asyncio soundfile

In [ ]:
# @title 3. Load & Cache Generative Music Model (MusicGen Small)
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Meta MusicGen model onto device: {device}...")

MODEL_ID = "facebook/musicgen-small"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = MusicgenForConditionalGeneration.from_pretrained(MODEL_ID).to(device)

print("MusicGen model loaded and cached in VRAM successfully!")

In [ ]:
# @title 4. Define DSP Mastery Node & Pitch Correction Pipeline
import io
import base64
import numpy as np
import scipy.signal
import soundfile as sf
import librosa
import pyworld as pw
import noisereduce as nr
from pedalboard import Pedalboard, HighpassFilter, PeakFilter, HighShelfFilter, Compressor, Reverb

def midi_to_hz(midi_note):
    """Convert MIDI note number to frequency in Hz (A4 = 440 Hz)."""
    return 440.0 * (2.0 ** ((midi_note - 69.0) / 12.0))

def hz_to_midi(freq):
    """Convert frequency in Hz to nearest floating-point MIDI note number."""
    return 69.0 + 12.0 * np.log2(np.maximum(freq, 1e-6) / 440.0)

def humanized_pitch_correction(audio_data, sr=32000, retune_strength=0.55):
    """
    Applies fractional pitch correction using PyWorld vocoder (DIO + StoneMask).
    Instead of rigid quantization, pulls pitch fractionally toward harmonic scale,
    preserving natural human vibrato and micro-dynamics.
    """
    x = audio_data.astype(np.float64)
    # Extract F0, spectral envelope, and aperiodicity
    _f0, t = pw.dio(x, sr, frame_period=5.0)
    f0 = pw.stonemask(x, _f0, t, sr)
    sp = pw.cheaptrick(x, f0, t, sr)
    ap = pw.d4c(x, f0, t, sr)
    
    # Smooth glitchy jumps using median filter
    smoothed_f0 = scipy.signal.medfilt(f0, kernel_size=5)
    corrected_f0 = np.copy(smoothed_f0)
    
    # Apply fractional retune to voiced frames
    voiced_indices = np.where(smoothed_f0 > 50.0)[0]
    for idx in voiced_indices:
        curr_f0 = smoothed_f0[idx]
        curr_midi = hz_to_midi(curr_f0)
        target_midi = np.round(curr_midi)  # Nearest chromatic semitone
        target_f0 = midi_to_hz(target_midi)
        # Fractional pull: preserve natural expression
        corrected_f0[idx] = curr_f0 + retune_strength * (target_f0 - curr_f0)
        
    # Resynthesize audio using modified F0
    y = pw.synthesize(corrected_f0, sp, ap, sr, frame_period=5.0)
    return y.astype(np.float32)

def audio_array_to_base64(audio_array, sr=32000):
    """Converts a float NumPy audio array to Base64-encoded WAV format."""
    buf = io.BytesIO()
    # Clip to valid audio range before writing
    clipped = np.clip(audio_array, -1.0, 1.0)
    sf.write(buf, clipped, sr, format="WAV", subtype="PCM_16")
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("utf-8")

def process_audio_pipeline(vocal_bytes, mood_prompt, duration_seconds=10):
    """
    Full DSP & Neural Pipeline:
    1. Synthesizes instrumental with MusicGen.
    2. Cleans raw vocals with noisereduce.
    3. Corrects vocal pitch with PyWorld (retune_strength=0.55).
    4. Shapes stems using Spotify Pedalboard EQ & compression.
    5. Mixes, normalizes, and soft-limits (np.tanh) final master.
    """
    TARGET_SR = 32000
    
    # --- 1. Synthesize Instrumental Backing Track ---
    print(f"Generating instrumental for prompt: \"{mood_prompt}\"")
    inputs = processor(text=[mood_prompt], padding=True, return_tensors="pt").to(device)
    
    # Calculate max_new_tokens for target duration (50 tokens per second)
    tokens_to_generate = int(duration_seconds * 50)
    with torch.no_grad():
        audio_values = model.generate(**inputs, max_new_tokens=tokens_to_generate)
    
    # Convert generated tensor to 1D numpy array
    raw_beat = audio_values[0, 0].cpu().numpy().astype(np.float32)
    
    # Instrumental EQ shaping
    beat_board = Pedalboard([
        HighpassFilter(cutoff_frequency_hz=40.0),
        PeakFilter(cutoff_frequency_hz=2500.0, gain_db=-2.0, q=0.7)
    ])
    shaped_beat = beat_board(raw_beat, TARGET_SR)
    
    # --- 2. Process Vocal Performance ---
    vocal_buf = io.BytesIO(vocal_bytes)
    vocal_data, in_sr = sf.read(vocal_buf)
    
    # Convert to mono if stereo
    if len(vocal_data.shape) > 1:
        vocal_data = np.mean(vocal_data, axis=1)
        
    # Resample to target sample rate
    if in_sr != TARGET_SR:
        vocal_data = librosa.resample(vocal_data.astype(np.float32), orig_sr=in_sr, target_sr=TARGET_SR)
        
    # Noise Reduction
    reduced_noise_vocal = nr.reduce_noise(y=vocal_data, sr=TARGET_SR, stationary=False)
    
    # Soft Pitch Correction
    pitch_corrected_vocal = humanized_pitch_correction(reduced_noise_vocal, sr=TARGET_SR, retune_strength=0.55)
    
    # Vocal FX Chain (Pedalboard)
    vocal_board = Pedalboard([
        HighpassFilter(cutoff_frequency_hz=100.0),
        PeakFilter(cutoff_frequency_hz=300.0, gain_db=-3.0, q=1.0),
        HighShelfFilter(cutoff_frequency_hz=7500.0, gain_db=2.5),
        Compressor(threshold_db=-22.0, ratio=4.0, attack_ms=10.0, release_ms=100.0),
        Reverb(room_size=0.25, wet_level=0.15, dry_level=0.85)
    ])
    mastered_vocal = vocal_board(pitch_corrected_vocal, TARGET_SR)
    
    # --- 3. Dynamic Mixbus Alignment & Mastering ---
    # Match lengths between beat and vocal
    target_len = max(len(shaped_beat), len(mastered_vocal))
    
    aligned_beat = np.zeros(target_len, dtype=np.float32)
    aligned_vocal = np.zeros(target_len, dtype=np.float32)
    
    aligned_beat[:len(shaped_beat)] = shaped_beat
    aligned_vocal[:len(mastered_vocal)] = mastered_vocal
    
    # Peak balancing (Beat target: 0.35, Vocal target: 0.90)
    max_beat = np.max(np.abs(aligned_beat)) + 1e-6
    max_vocal = np.max(np.abs(aligned_vocal)) + 1e-6
    
    norm_beat = (aligned_beat / max_beat) * 0.35
    norm_vocal = (aligned_vocal / max_vocal) * 0.90
    
    # Mix summation and soft-clipping saturation via np.tanh
    summed_mix = norm_beat + norm_vocal
    master_track = np.tanh(summed_mix * 1.1) * 0.95
    
    # --- 4. Export Base64 Stems ---
    return {
        "master_track": audio_array_to_base64(master_track, TARGET_SR),
        "music_track": audio_array_to_base64(norm_beat, TARGET_SR),
        "vocal_track": audio_array_to_base64(norm_vocal, TARGET_SR),
        "status": "success"
    }

print("DSP and mastering pipeline defined successfully.")

In [ ]:
# @title 5. Define FastAPI GPU Server
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="IsaiCraft GPU Processing Engine", version="1.0.0")

# Configure CORS for local React development
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
async def health_check():
    return {"status": "online", "device": device, "engine": "IsaiCraft Neural Node v1.0"}

@app.post("/generate")
async def generate_music(
    project_name: str = Form("Unnamed"),
    mood: str = Form("Dynamic performance"),
    genre: str = Form("Unknown"),
    lyrics: str = Form(""),
    vocal_file: UploadFile = File(...)
):
    try:
        vocal_bytes = await vocal_file.read()
        if not vocal_bytes:
            raise HTTPException(status_code=400, detail="Empty audio file received")
            
        prompt = f"{genre} track with {mood}. Lyrical theme: {lyrics[:100]}"
        result = process_audio_pipeline(vocal_bytes, prompt, duration_seconds=10)
        return result
    except Exception as e:
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

In [ ]:
# @title 6. Launch Secure HTTP Tunnel & Start Processing Server
import nest_asyncio
import uvicorn
from pyngrok import ngrok, conf
import getpass

# Safe ngrok token entry (interactive prompt or Google Colab Secret)
print("Enter your ngrok authtoken (sign up for free at https://dashboard.ngrok.com):")
try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    NGROK_TOKEN = None

if not NGROK_TOKEN:
    NGROK_TOKEN = getpass.getpass("Ngrok Token (hidden): ")

if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN

# Terminate existing tunnels and open port 8000
ngrok.kill()
public_url = ngrok.connect(8000)
print("\n" + "="*60)
print(f"🚀 ISAICRAFT GPU ENGINE LIVE AT: {public_url.public_url}")
print(f"👉 Copy this URL and set VITE_GPU_ENGINE_URL in frontend/.env")
print("="*60 + "\n")

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)